<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 100.2 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Imports**

In [4]:
!pip install pytorch_forecasting -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 33.6 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import torch
from lightning.pytorch.loggers import WandbLogger

import json


In [6]:
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

**Data Processing**

In [7]:
train_df = pd.read_csv("/content/drive/MyDrive/ColabNotebooks/train_processed.csv")
val_df = pd.read_csv("/content/drive/MyDrive/ColabNotebooks/val_processed.csv")

In [8]:
val_df.columns

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature',
       'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4',
       'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size', 'Week', 'Month',
       'Year', 'Quarter', 'holiday_name', 'lag_1', 'lag_2', 'lag_4', 'lag_52',
       'rolling_mean_4', 'rolling_mean_12', 'rolling_std_4',
       'rolling_mean_52'],
      dtype='object')

**Preprocessing**

In [9]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class PickColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_keep=None):
        self.columns_to_keep = [
            'Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature',
            'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4',
            'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size'
            ,'Week', 'Month', 'Year', 'Quarter', 'holiday_name'
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X[self.columns_to_keep].copy()

In [10]:
column_picker = PickColumns()
X_train = column_picker.fit_transform(train_df)
X_val = column_picker.transform(val_df)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332778 entries, 0 to 332777
Data columns (total 21 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         332778 non-null  int64  
 1   Dept          332778 non-null  int64  
 2   Date          332778 non-null  object 
 3   Weekly_Sales  332778 non-null  float64
 4   IsHoliday     332778 non-null  bool   
 5   Temperature   332778 non-null  float64
 6   Fuel_Price    332778 non-null  float64
 7   MarkDown1     332778 non-null  float64
 8   MarkDown2     332778 non-null  float64
 9   MarkDown3     332778 non-null  float64
 10  MarkDown4     332778 non-null  float64
 11  MarkDown5     332778 non-null  float64
 12  CPI           332778 non-null  float64
 13  Unemployment  332778 non-null  float64
 14  Type          332778 non-null  object 
 15  Size          332778 non-null  int64  
 16  Week          332778 non-null  int64  
 17  Month         332778 non-null  int64  
 18  Year

In [11]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class ConvertToCategorical(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns or ['Store', 'Dept', 'IsHoliday', 'Type',  'holiday_name']

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columns_ = list(self.columns)

        missing = set(self.columns_) - set(X.columns)
        if missing:
            raise ValueError(f"Missing expected columns: {missing}")

        self.categories_ = {
            col: pd.Series(X[col].astype(str).unique()).sort_values().tolist()
            for col in self.columns_
        }
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in self.columns_:
            X[col] = pd.Categorical(X[col].astype(str), categories=self.categories_[col])
        return X

In [12]:
class DateToTimeIdx(BaseEstimator, TransformerMixin):

    def __init__(self, date_col='Date', time_idx_col='time_idx'):
        self.date_col = date_col
        self.time_idx_col = time_idx_col

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        dates = pd.to_datetime(X[self.date_col])
        self.min_date_ = dates.min()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X[self.date_col] = pd.to_datetime(X[self.date_col])

        delta_days = (X[self.date_col] - self.min_date_).dt.days
        X[self.time_idx_col] = (delta_days // 7).astype(int)

        return X


In [13]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('pick_columns', PickColumns()),
    ('to_categorical', ConvertToCategorical()),
    ('Date_processor', DateToTimeIdx())
])
X_train = pipeline.fit_transform(train_df)
X_val = pipeline.transform(val_df)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332778 entries, 0 to 332777
Data columns (total 22 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         332778 non-null  category      
 1   Dept          332778 non-null  category      
 2   Date          332778 non-null  datetime64[ns]
 3   Weekly_Sales  332778 non-null  float64       
 4   IsHoliday     332778 non-null  category      
 5   Temperature   332778 non-null  float64       
 6   Fuel_Price    332778 non-null  float64       
 7   MarkDown1     332778 non-null  float64       
 8   MarkDown2     332778 non-null  float64       
 9   MarkDown3     332778 non-null  float64       
 10  MarkDown4     332778 non-null  float64       
 11  MarkDown5     332778 non-null  float64       
 12  CPI           332778 non-null  float64       
 13  Unemployment  332778 non-null  float64       
 14  Type          332778 non-null  category      
 15  Size          332

**Creating Dataset**

In [14]:
from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer

max_encoder_length = 52
max_prediction_length = 35

training = TimeSeriesDataSet(
    X_train,
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["Store", "Dept"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,


    static_categoricals=["Store", "Dept", "Type"],
    static_reals=["Size"],

    time_varying_known_categoricals=["IsHoliday", "holiday_name"],
    time_varying_known_reals=[
        "time_idx", "Year", "Month", "Week", "Quarter",
        "Temperature", "Fuel_Price", "CPI", "Unemployment",
        "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    ],


    time_varying_unknown_reals=["Weekly_Sales"],

    target_normalizer=GroupNormalizer(
        groups=["Store", "Dept"]
    ),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,

    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    pd.concat([X_train, X_val], axis=0).reset_index(drop=True),
    predict=True,
    stop_randomization=True,
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 291 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '1', '__group_id__Dept': '51'}, {'__group_id__Store': '1', '__group_id__Dept': '77'}, {'__group_id__Store': '1', '__group_id__Dept': '78'}, {'__group_id__Store': '10', '__group_id__Dept': '51'}, {'__group_id__Store': '10', '__group_id__Dept': '77'}, {'__group_id__Store': '11', '__group_id__Dept': '48'}, {'__group_id__Store': '11', '__group_id__Dept': '50'}, {'__group_id__Store': '11', '__group_id__Dept': '51'}, {'__group_id__Store': '11', '__group_id__Dept': '77'}, {'__group_id__Store': '11', '__group_id__Dept': '78'}]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/tim

In [15]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332778 entries, 0 to 332777
Data columns (total 22 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         332778 non-null  category      
 1   Dept          332778 non-null  category      
 2   Date          332778 non-null  datetime64[ns]
 3   Weekly_Sales  332778 non-null  float64       
 4   IsHoliday     332778 non-null  category      
 5   Temperature   332778 non-null  float64       
 6   Fuel_Price    332778 non-null  float64       
 7   MarkDown1     332778 non-null  float64       
 8   MarkDown2     332778 non-null  float64       
 9   MarkDown3     332778 non-null  float64       
 10  MarkDown4     332778 non-null  float64       
 11  MarkDown5     332778 non-null  float64       
 12  CPI           332778 non-null  float64       
 13  Unemployment  332778 non-null  float64       
 14  Type          332778 non-null  category      
 15  Size          332

In [16]:
train_loader = training.to_dataloader(train=True, batch_size=64, num_workers=0)
val_loader = validation.to_dataloader(train=False, batch_size=64, num_workers=0)

**Loss**

In [109]:
# WMAE metric
import numpy as np

def WMAE(y_true, y_pred, is_holiday):
    is_holiday_bool = is_holiday.astype(str).str.lower().isin(["true"])
    weights = np.where(is_holiday_bool, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

**Baseline**

In [67]:
from pytorch_forecasting import Baseline

baseline_predictions = Baseline().predict(
    val_loader,
    return_x=True,
    return_y=True,
    return_index=True,
)

y_pred = baseline_predictions.output.cpu()
y_true = baseline_predictions.y[0].cpu()
index_df = baseline_predictions.index  # already a dataframe with Store, Dept, time_idx per row

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

In [68]:
y_pred.shape, y_true.shape, index_df.shape

(torch.Size([2947, 35]), torch.Size([2947, 35]), (2947, 3))

In [102]:
import numpy as np
import pandas as pd

# y_pred, y_true shape: (num_samples, prediction_length)
pred_len = y_pred.shape[1]

# Repeat each index row pred_len times, and add offset 0..pred_len-1 to time_idx
index_expanded = index_df.loc[index_df.index.repeat(pred_len)].reset_index(drop=True)
index_expanded["time_idx"] = (
    index_expanded["time_idx"].values
    + np.tile(np.arange(pred_len), len(index_df))
)

# Flatten predictions/targets to match row-by-row
index_expanded["y_pred"] = y_pred.numpy().flatten()
index_expanded["y_true"] = y_true.numpy().flatten()
full_df = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)

# Now merge with your full dataset on the keys + time_idx
print(index_expanded.shape)
merged = index_expanded.merge(
    full_df,
    on=["Store", "Dept", "time_idx"],
    how="inner"   # or "inner" if you only want rows that have predictions
)
merged.info()

(103145, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100795 entries, 0 to 100794
Data columns (total 24 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   time_idx      100795 non-null  int64         
 1   Store         100795 non-null  object        
 2   Dept          100795 non-null  object        
 3   y_pred        100795 non-null  float32       
 4   y_true        100795 non-null  float32       
 5   Date          100795 non-null  datetime64[ns]
 6   Weekly_Sales  100795 non-null  float64       
 7   IsHoliday     100795 non-null  category      
 8   Temperature   100795 non-null  float64       
 9   Fuel_Price    100795 non-null  float64       
 10  MarkDown1     100795 non-null  float64       
 11  MarkDown2     100795 non-null  float64       
 12  MarkDown3     100795 non-null  float64       
 13  MarkDown4     100795 non-null  float64       
 14  MarkDown5     100795 non-null  float64       
 15  CPI  

In [103]:
print(merged["y_true"].isna().sum())
print(merged["y_pred"].isna().sum())
print(merged["IsHoliday"].isna().sum())
print(len(merged))

0
0
0
100795


In [110]:
wmae = WMAE(merged["y_true"], merged["y_pred"], merged["IsHoliday"])
print(f"WMAE: {wmae}")

WMAE: 2801.958783269697
